# Tutorial 5, Q2: fire stations, set covering

Release after the practical.

**The problem.** Table 2 of the sheet lists the cities that a fire station in each city would cover. Build as few
stations as possible so that every city is covered. $x_i = 1$ if a station is built in city $i$.

| Station in city | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Covers cities | 1, 2, 3 | 1, 2, 4 | 1, 3 | 2, 4, 5 | 4, 5, 6 | 5, 6, 7 | 6, 7 |

For the constraints, read the table the other way round: city 1 is covered by a station in city 1, 2 or 3, so
$x_1 + x_2 + x_3 \ge 1$; and so on for each city. minimise $\sum_{i=1}^{7} x_i$, $x_i \in \{0, 1\}$.
In the code, cities 1–7 are `x[0]` … `x[6]`.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** **3 stations**. The browser shows cities 1, 2 and 6 (`x[0]`, `x[1]`, `x[5]` = 1); the answer
sheet shows cities 3, 4 and 7. Both cover every city: the problem has several optimal answers.

*Original script: `5 Source Files/Python/firestation_tutorial.py`. Runs in the browser with HiGHS instead of GLPK; the model and the optimal value are unchanged.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

A *concrete model* is the container for the problem.

In [ ]:
# Create a concrete model
model = ConcreteModel()

One binary variable per city: build a station there (1) or not (0).

In [ ]:
# Define the binary variables: x[i] = 1 if a fire station is built in city i+1
model.x = Var(range(7), within=Binary)

The objective: the number of stations, minimised.

In [ ]:
# Define the objective function
model.obj = Objective(expr=sum(model.x[i] for i in range(7)), sense=minimize)

One constraint per city: at least one of the stations that cover it is built.

In [ ]:
# Define the inequality constraints: every city is covered by at least one station
model.city1 = Constraint(expr=model.x[0] + model.x[1] + model.x[2] >= 1)  # city 1: station in 1, 2 or 3
model.city2 = Constraint(expr=model.x[0] + model.x[1] + model.x[3] >= 1)  # city 2: station in 1, 2 or 4
model.city3 = Constraint(expr=model.x[0] + model.x[2] >= 1)               # city 3: station in 1 or 3
model.city4 = Constraint(expr=model.x[1] + model.x[3] + model.x[4] >= 1)  # city 4: station in 2, 4 or 5
model.city5 = Constraint(expr=model.x[3] + model.x[4] + model.x[5] >= 1)  # city 5: station in 4, 5 or 6
model.city6 = Constraint(expr=model.x[4] + model.x[5] + model.x[6] >= 1)  # city 6: station in 5, 6 or 7
model.city7 = Constraint(expr=model.x[5] + model.x[6] >= 1)               # city 7: station in 6 or 7

Create the HiGHS solver, print the model (`pprint()`: check it against the problem above), solve, and print the results (`display()`).

The HiGHS log appears between the two printouts (in the browser it is printed even without `tee=True`); its *Solving report* should say *Status Optimal*. In `display()`, look at the `Value` column of the variables and at the objective.

In [ ]:
# Create a solver
solver = SolverFactory('appsi_highs')
# Display the model
model.pprint()
# Solve the model
solver.solve(model)
# Display the results
model.display()

## What do you notice?

Why can two stations never be enough?

<details><summary>Answer</summary>

City 3 can only be covered from city 1 or 3, and city 7 only from city 6 or 7. A station in 1 or 3 covers at
most cities 1, 2, 3; a station in 6 or 7 covers at most cities 5, 6, 7. City 4 is left, so a third station is
needed.
</details>